# Circuit Simplification - Local Execution

This notebook demonstrates running circuit simplification locally using `obi-one`.

In production, jobs are launched via TaskManager. This notebook lets you run and debug simplification locally before submitting to the task queue.

**Two modes:**
1. **From entitycore ID** - Fetches and stages the circuit from entitycore (production workflow)
2. **From local path** - Uses a local circuit file (for testing)

**Prerequisites:**
- `obi-one` installed
- `sonata_simplify` installed  
- NEURON with compiled mod files (for `single_compartment` algorithm)

**Writes to:** `obi-output/circuit_simplification/` containing:
- `obi_one_coordinate.json` - the TaskConfig
- `<algorithm>/output/circuit_config.json` - the simplified circuit

## Imports

In [ ]:
import json
from pathlib import Path

import obi_one as obi
from entitysdk import Client, ProjectContext
from obi_auth import get_token
from obi_one.core.info import Info
from obi_one.db_sdk import db_sdk
from obi_one.scientific.from_id.circuit_from_id import CircuitFromID
from obi_one.scientific.library.circuit import Circuit
from obi_one.scientific.blocks.simplification_algorithms import (
    SingleCompartmentAlgorithm,
)
from obi_one.scientific.tasks.circuit_simplification import (
    CircuitSimplificationScanConfig,
    CircuitSimplificationTask,
)

## Connect to entitycore staging

In [ ]:
virtual_lab_id = obi.LAB_ID_STAGING_TEST
project_id = obi.PROJECT_ID_STAGING_TEST

token = get_token(environment="staging")
project_context = ProjectContext(virtual_lab_id=virtual_lab_id, project_id=project_id)
db_client = Client(
    api_url="https://staging.openbraininstitute.org/api/entitycore",
    project_context=project_context,
    token_manager=token,
)
print("Connected to entitycore staging.")

## Configuration

Choose one of the two modes:

**Mode 1: From entitycore ID** (production workflow)  
Set `circuit_id` to the UUID of a circuit in entitycore.

**Mode 2: From local path** (for testing)  
Set `circuit_id = None` and provide a local `circuit_path`.

In [ ]:
# ============================================================
# MODE 1: From entitycore ID (set circuit_id to a UUID string)
# ============================================================
# circuit_id = "4733054a-edc5-46cb-80cd-748758879355"  # nbS1-O1-sSub-pre-dim5-nCN-HEX0-L6-01 # smalll microcircuit
circuit_id = "ed096991-4171-4cfc-b54e-756323fc62e7" # nbS1-O1-E2Sst-maxNsyn-HEX0-L3 # paired neuron
# ============================================================
# MODE 2: From local path (set circuit_id = None)
# ============================================================
# circuit_id = None
# local_circuit_path = Path("../../../../data/tiny_circuits/N_10__top_nodes_dim6/circuit_config.json")

# Output directory
output_root = Path("../../../../../../obi-output/circuit_simplification")
output_root = output_root.resolve()
output_root.mkdir(parents=True, exist_ok=True)

print(f"Output root: {output_root}")

## Clean Up Old Outputs

Remove previous simplification runs to avoid stale results interfering with the new run.  
Set `clean_output = True` to delete all contents of the output root (except the entity cache).

In [ ]:
import shutil

clean_output = True  # Set to False to keep old runs

if clean_output:
    entity_cache = output_root / "entity_cache"
    for item in output_root.iterdir():
        if item == entity_cache:
            continue
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()
    print(f"Cleaned output root (kept entity_cache): {output_root}")
else:
    print(f"Keeping existing outputs in: {output_root}")

## Create Circuit Reference

If using a circuit ID, this creates a `CircuitFromID` reference.  
If using a local path, this creates a `Circuit` object directly.

In [ ]:
if circuit_id is not None:
    # Mode 1: From entitycore ID
    circuit_ref = CircuitFromID(id_str=circuit_id)
    
    # Fetch entity info to display
    circuit_entity = circuit_ref.entity(db_client=db_client)
    print(f"Circuit from entitycore:")
    print(f"  ID: {circuit_id}")
    print(f"  Name: {circuit_entity.name}")
    print(f"  Description: {circuit_entity.description[:200]}..." if len(circuit_entity.description or '') > 200 else f"  Description: {circuit_entity.description}")
else:
    # Mode 2: From local path
    circuit_path = local_circuit_path.resolve()
    if not circuit_path.exists():
        raise FileNotFoundError(f"Circuit config not found: {circuit_path}")
    
    circuit_ref = Circuit(name="local_circuit", path=str(circuit_path))
    print(f"Local circuit: {circuit_path}")

## Stage Circuit (for entitycore circuits)

When using a circuit ID, we need to download (stage) the circuit from entitycore.  
This step downloads the circuit assets to a local cache directory.

**Note:** This may take a while for large circuits. If it fails with a network error, retry this cell.

In [ ]:
import tempfile

if circuit_id is not None:
    print(f"Staging circuit from entitycore...")
    print(f"  Circuit ID: {circuit_id}")
    print("  (This may take a while for large circuits)")
    print()

    # Stage into a persistent cache directory so retries don't re-download.
    # temp_dir is required by the API even when entity_cache=True.
    with tempfile.TemporaryDirectory() as tmp:
        circuit, circuit_entity = db_sdk.resolve_circuit(
            circuit_ref,
            db_client=db_client,
            entity_cache=True,
            cache_root=output_root,
            temp_dir=Path(tmp),
        )

    print(f"Circuit staged successfully!")
    print(f"  Local path: {circuit.path}")
else:
    # Local circuit - no staging needed
    circuit = circuit_ref
    circuit_entity = None
    print("Using local circuit - no staging needed.")


## Compile NEURON Mechanisms (for `single_compartment` algorithm)

The `single_compartment` algorithm requires NEURON with the circuit's MOD files compiled.  
This step runs `nrnivmodl` on the staged circuit's `mod/` directory.

**Skip this cell if:** using only point-neuron algorithms (`lif`, `adex`, `izhikevich`, `glif`, `gif`) without `single_compartment`.

In [ ]:
import platform
import subprocess
import sys

circuit_dir = Path(circuit.path).parent
mod_dir = circuit_dir / "mod"
arch = "arm64" if platform.machine() == "arm64" else "x86_64"
compiled_dir = circuit_dir / arch

if mod_dir.exists():
    # Check for empty .mod files (data corruption from staging)
    empty_mods = [f for f in mod_dir.glob("*.mod") if f.stat().st_size == 0]
    if empty_mods:
        print(f"Found {len(empty_mods)} empty MOD file(s):")
        for f in empty_mods:
            print(f"  - {f.name}")
        # Try to find valid copies from the tiny_circuits examples
        fallback_dirs = [
            Path("../../data/tiny_circuits/N_10__top_nodes_dim6/mod"),
            Path("../../data/tiny_circuits/nbS1-O1-E2Sst-maxNsyn-HEX0-L5/mod"),
        ]
        for empty_mod in empty_mods:
            found = False
            for fallback in fallback_dirs:
                candidate = fallback.resolve() / empty_mod.name
                if candidate.exists() and candidate.stat().st_size > 0:
                    import shutil
                    shutil.copy2(candidate, empty_mod)
                    print(f"  Replaced {empty_mod.name} from {candidate}")
                    found = True
                    break
            if not found:
                print(f"  WARNING: No valid copy found for {empty_mod.name}, removing")
                empty_mod.unlink()

    if not compiled_dir.exists() or not (compiled_dir / "libnrnmech.dylib").exists():
        print(f"Compiling MOD files in {mod_dir}...")
        nrnivmodl = Path(sys.prefix) / "bin" / "nrnivmodl"
        if not nrnivmodl.exists():
            nrnivmodl = "nrnivmodl"
        result = subprocess.run(
            [str(nrnivmodl), "-incflags", "-DDISABLE_REPORTINGLIB", str(mod_dir)],
            cwd=circuit_dir,
            capture_output=True,
            text=True,
        )
        if result.returncode != 0:
            print(f"nrnivmodl failed (exit {result.returncode}):")
            print(result.stderr[-2000:] if result.stderr else "(no stderr)")
        else:
            print(f"MOD files compiled successfully: {compiled_dir}")
    else:
        print(f"Mechanisms already compiled: {compiled_dir}")
else:
    print(f"No mod/ directory found at {mod_dir}")
    print("Skipping compilation (may not be needed for point-neuron algorithms)")

## Inspect Circuit

In [ ]:
# Inspect the circuit
sonata = circuit.sonata_circuit
print(f"Circuit: {circuit.name}")
print(f"Path: {circuit.path}")
print(f"\nNode populations:")
for pop_name in sonata.nodes.population_names:
    pop = sonata.nodes[pop_name]
    print(f"  - {pop_name}: {pop.size} neurons")

print(f"\nEdge populations:")
for pop_name in sonata.edges.population_names:
    pop = sonata.edges[pop_name]
    print(f"  - {pop_name}: {pop.size} synapses")

In [ ]:
# Show etype distribution (important for simplification parameters)
for pop_name in sonata.nodes.population_names:
    pop = sonata.nodes[pop_name]
    try:
        df = pop.get()
        if "etype" in df.columns:
            print(f"\nEtype distribution for {pop_name}:")
            print(df["etype"].value_counts())
    except Exception as e:
        print(f"Could not get etype distribution for {pop_name}: {e}")

## Build the Scan Config

Available algorithm blocks (add one or more):
- `single_compartment`: Single-compartment model (NEURON)
- `lif_nest`: Leaky integrate-and-fire (NEST)
- `adex_nest`: Adaptive exponential integrate-and-fire (NEST)
- `adex_brian2`: Adaptive exponential integrate-and-fire (Brian2)
- `izhikevich_nest`: Izhikevich model (NEST)
- `glif_nest`: Generalized leaky integrate-and-fire (NEST)
- `gif_nest`: Generalized integrate-and-fire (NEST)

The `algorithms` field is a root-level `block_dictionary`. Each selected block produces a separate simplified output circuit.
Algorithms with NEST/Brian2 exports additionally produce separate Circuit entities with the appropriate `target_simulator`.
The block cards currently contain descriptions only and can be extended with algorithm-specific parameters in the future.

In [ ]:
# Use the staged circuit (local Circuit object) for the scan config
# This avoids re-staging during task execution
scan_config = CircuitSimplificationScanConfig(
    info=Info(
        campaign_name="Circuit Simplification Test",
        campaign_description="Testing circuit simplification locally",
    ),
    initialize=CircuitSimplificationScanConfig.Initialize(
        circuit=circuit,  # Use the staged Circuit, not CircuitFromID
    ),
    algorithms={
        "single_compartment": SingleCompartmentAlgorithm(),
    },
)

print("Scan config created.")
print(f"  Circuit: {circuit.name}")
print(f"  Algorithms: {scan_config.algorithms}")

## Generate TaskConfig

In [ ]:
grid_scan = obi.GridScanGenerationTask(
    form=scan_config,
    output_root=str(output_root),
    coordinate_directory_option="ZERO_INDEX",
)
grid_scan.execute()

coord_root = Path(grid_scan.single_configs[0].coordinate_output_root).resolve()
print(f"Coordinate output root: {coord_root}")
print(f"Generated {len(grid_scan.single_configs)} task config(s)")

## Run the Simplification Task

This is the main execution step. The task will:
1. Run the simplification pipeline on the staged circuit
2. Optionally register the output circuit entity in entitycore

**Note:** The simplification can take several minutes depending on circuit size.

In [ ]:
single_config = grid_scan.single_configs[0]
task = CircuitSimplificationTask(config=single_config)

print("Running simplification task...")
print("(This may take several minutes for filter computation)")

# Pass db_client to enable registering output circuits to entitycore
# Set db_client=None for local-only execution without registration
result = task.execute(db_client=db_client)

print(f"\nSimplification complete!")
if result:
    print(f"Registered circuit entity ID: {result}")
else:
    print("No circuit was registered (local-only execution or registration disabled)")

## Inspect Results

In [ ]:
# Find generated simplified circuits
# The task writes to: <coordinate_output_root>/<algorithm>/output/circuit_config.json
# Exclude entity_cache (which contains the original circuit_config.json)
simplified_configs = [
    p for p in output_root.glob("**/output/circuit_config.json")
    if "entity_cache" not in p.parts
]
print(f"Found {len(simplified_configs)} simplified circuit(s):")
for cfg in simplified_configs:
    print(f"  - {cfg}")

In [ ]:
# Load and compare original vs simplified
from bluepysnap import Circuit as SnapCircuit

if simplified_configs:
    simplified_circuit = SnapCircuit(str(simplified_configs[0]))
    
    print("=== Original Circuit ===")
    for pop_name in sonata.nodes.population_names:
        print(f"{pop_name}: {sonata.nodes[pop_name].size} neurons")
    
    print("\n=== Simplified Circuit ===")
    for pop_name in simplified_circuit.nodes.population_names:
        print(f"{pop_name}: {simplified_circuit.nodes[pop_name].size} neurons")
else:
    print("No simplified circuits found. Check logs for errors.")

In [ ]:
# Show simplified neuron properties
if simplified_configs:
    for pop_name in simplified_circuit.nodes.population_names:
        pop = simplified_circuit.nodes[pop_name]
        df = pop.get()
        print(f"\n=== {pop_name} properties ===")
        # Show relevant columns for simplified neurons
        display_cols = [c for c in df.columns if c in [
            'etype', 'mtype', 'model_type', 'model_template',
            'x', 'y', 'z', '@dynamics:holding_current'
        ]]
        if display_cols:
            print(df[display_cols].head(10))
        else:
            print(df.head(10))

## Export to NEST and Brian2 Formats

The `sonata_simplify` library provides exporters that convert simplified SONATA circuits into NEST-compatible and Brian2-compatible formats. Each export produces a self-contained `circuit_config.json` in the output directory.

**NEST export** requires `nest-simulator` (install with `uv pip install nest-simulator`; supports Python 3.9–3.14).  
**Brian2 export** works with the standard `obi-one` environment (Brian2 is installed).

In [ ]:
from sonata_simplify.exporters import get_exporter, list_exporters

print("Available exporters:")
for name in list_exporters():
    print(f"  - {name}")

# Use the first simplified circuit from the results
simplified_dir = simplified_configs[0].parent if simplified_configs else None

if simplified_dir is None:
    raise FileNotFoundError("No simplified circuit found. Run the simplification task first.")

print(f"\nSimplified circuit: {simplified_dir}")

# Export to NEST (aeif_cond_alpha model)
nest_output = output_root / "export_nest"
nest_output.mkdir(parents=True, exist_ok=True)
nest_exporter = get_exporter(
    "nest:aeif_cond_alpha",
    input_circuit_dir=simplified_dir,
    output_dir=nest_output,
)
nest_result = nest_exporter.export()
print(f"\nNEST export: {nest_result}")
print(f"  circuit_config.json: {(nest_result / 'circuit_config.json').exists()}")

# Export to Brian2 (adex model)
brian2_output = output_root / "export_brian2"
brian2_output.mkdir(parents=True, exist_ok=True)
brian2_exporter = get_exporter(
    "brian2:adex",
    input_circuit_dir=simplified_dir,
    output_dir=brian2_output,
)
brian2_result = brian2_exporter.export()
print(f"\nBrian2 export: {brian2_result}")
print(f"  circuit_config.json: {(brian2_result / 'circuit_config.json').exists()}")

In [ ]:
# Compare the exported circuit configs
import json

for label, cfg_path in [
    ("SONATA (NEURON)", simplified_dir / "circuit_config.json"),
    ("NEST", nest_result / "circuit_config.json"),
    ("Brian2", brian2_result / "circuit_config.json"),
]:
    with open(cfg_path) as f:
        cfg = json.load(f)
    n_nodes = len(cfg.get("networks", {}).get("nodes", []))
    n_edges = len(cfg.get("networks", {}).get("edges", []))
    target_sim = cfg.get("target_simulator", "NEURON (default)")
    print(f"\n=== {label} ===")
    print(f"  target_simulator: {target_sim}")
    print(f"  node populations: {n_nodes}")
    print(f"  edge populations: {n_edges}")
    for node in cfg.get("networks", {}).get("nodes", []):
        print(f"    nodes: {node.get('nodes_file', '?')}")
    for edge in cfg.get("networks", {}).get("edges", []):
        print(f"    edges: {edge.get('edges_file', '?')}")

## Run Brian2 Simulation on Exported Circuit

This runs a simple DC stimulus simulation on the Brian2-exported circuit and compares the results with the NEURON simplified circuit.

In [ ]:
import subprocess

brian2_results_dir = output_root / "brian2_simulation_results"
brian2_results_dir.mkdir(parents=True, exist_ok=True)

print("Running Brian2 simulation on exported circuit...")
print(f"  Circuit: {brian2_result / 'circuit_config.json'}")
print(f"  Output:  {brian2_results_dir}")

result = subprocess.run(
    [
        sys.executable, "-m", "sonata_simplify.brian2_simulation",
        "--circuit-config", str(brian2_result / "circuit_config.json"),
        "--output-dir", str(brian2_results_dir),
        "--tstop", "500.0",
        "--dt", "0.1",
        "--stim-amp", "400.0",
    ],
    capture_output=True,
    text=True,
)

print(f"\nExit code: {result.returncode}")
if result.stdout:
    for line in result.stdout.strip().split("\n"):
        print(f"  {line}")
if result.stderr:
    print(f"\nStderr (last 500 chars):")
    print(result.stderr[-500:])

In [ ]:
# Load and display Brian2 simulation results
brian2_results_path = brian2_results_dir / "brian2_results.json"
brian2_summary_path = brian2_results_dir / "simulation_summary.json"

if brian2_results_path.exists():
    with open(brian2_results_path) as f:
        brian2_results = json.load(f)
    
    print("=== Brian2 Simulation Results ===")
    for pop_name, pop_data in brian2_results.get("spikes", {}).items():
        n_spikes = len(pop_data.get("times", []))
        n_active = len(set(pop_data.get("indices", [])))
        print(f"  {pop_name}: {n_spikes} spikes from {n_active} neurons")
    
    # Show voltage trace info
    for pop_name, trace_data in brian2_results.get("voltage_traces", {}).items():
        n_traced = len(trace_data)
        print(f"  {pop_name} voltage: {n_traced} neurons recorded")
else:
    print("No Brian2 results found. Check simulation output above.")

if brian2_summary_path.exists():
    with open(brian2_summary_path) as f:
        summary = json.load(f)
    print(f"\n=== Simulation Summary ===")
    for key, val in summary.items():
        print(f"  {key}: {val}")

## Run NEST Simulation on Exported Circuit

NEST (`nest-simulator`) supports Python 3.9–3.14 and can be installed directly in the obi-one venv:
```bash
uv pip install nest-simulator
```
If NEST is not available, this cell will be skipped.

In [ ]:
# Check if NEST is available
# NEST (nest-simulator) supports Python 3.9-3.14 and can be installed
# directly in the obi-one venv: uv pip install nest-simulator
nest_available = False
nest_python = None

# Try the current Python first (NEST may be installed in this venv),
# then fall back to system Python candidates.
candidates = [sys.executable, "python3.14", "python3"]
for candidate in candidates:
    try:
        check = subprocess.run(
            [candidate, "-c", "import nest; import h5py; print('OK')"],
            capture_output=True, text=True, timeout=10,
        )
        if check.returncode == 0 and "OK" in check.stdout:
            nest_available = True
            nest_python = candidate
            break
    except (FileNotFoundError, subprocess.TimeoutExpired):
        continue

if nest_available:
    nest_results_dir = output_root / "nest_simulation_results"
    nest_results_dir.mkdir(parents=True, exist_ok=True)

    print(f"Running NEST simulation with {nest_python}...")
    result = subprocess.run(
        [
            nest_python, "-m", "sonata_simplify.nest_simulation",
            "--circuit-config", str(nest_result / "circuit_config.json"),
            "--output-dir", str(nest_results_dir),
            "--tstop", "500.0",
            "--dt", "0.1",
            "--stim-amp", "400.0",
        ],
        capture_output=True,
        text=True,
    )

    print(f"\nExit code: {result.returncode}")
    if result.stdout:
        for line in result.stdout.strip().split("\n")[-20:]:
            print(f"  {line}")
    if result.stderr:
        print(f"\nStderr (last 500 chars):")
        print(result.stderr[-500:])
else:
    print("NEST is not available on this machine.")
    print("To run NEST simulations, install NEST in this environment:")
    print("  uv pip install nest-simulator")
    print("(NEST supports Python 3.9-3.14; no separate venv needed)")
    print("Then re-run this cell.")

## Simulation Results: Comparison Plots

This cell loads all available simulation results and generates:

1. **Summary table** — spike counts, active neurons, mean firing rate per simulator
2. **Voltage traces** — membrane potential over time for each simulator (side-by-side)
3. **Spike raster** — spike times across neurons, color-coded by simulator
4. **Overlay comparison** — NEST vs Brian2 voltage traces on the same axes (per neuron)

Plots are saved as PNGs to the output root.

**Available results:**
- **Brian2** — from `brian2_simulation_results/`
- **NEST** — from `nest_simulation_results/`
- **Detailed NEURON** — not run (would require bluecellulab simulation on the original circuit)
- **Single-compartment NEURON** — not run (would require bluecellulab simulation on the simplified circuit)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use("module://matplotlib_inline.backend_inline")
import numpy as np
import pandas as pd

# ── Load all available simulation results ──────────────────────────────
sim_results = {}  # {simulator_name: {pop_name: pop_data}}

# Brian2
brian2_results_path = brian2_results_dir / "brian2_results.json"
if brian2_results_path.exists():
    with open(brian2_results_path) as f:
        brian2_data = json.load(f)
    for pop_name, pop_data in brian2_data.get("populations", {}).items():
        sim_results.setdefault("Brian2", {})[pop_name] = pop_data

# NEST
nest_results_path = output_root / "nest_simulation_results" / "nest_results.json"
if nest_results_path.exists():
    with open(nest_results_path) as f:
        nest_data = json.load(f)
    for pop_name, pop_data in nest_data.get("populations", {}).items():
        sim_results.setdefault("NEST", {})[pop_name] = pop_data

# NEURON (detailed / single_compartment) — not run in this notebook
# These would require bluecellulab + NEURON simulation on the SONATA circuit.
# The simulation_config.json exists in the simplified output but was not executed.

if not sim_results:
    print("No simulation results available for plotting.")
else:
    # ── 1. Summary table ──────────────────────────────────────────────
    print("=" * 70)
    print("SIMULATION SUMMARY")
    print("=" * 70)
    rows = []
    for sim_name, pops in sim_results.items():
        for pop_name, pdata in pops.items():
            n_nodes = pdata.get("n_nodes", 0)
            n_spikes = pdata.get("n_spikes", 0)
            n_active = len(set(pdata.get("spike_senders", [])))
            tstop = brian2_data.get("tstop", nest_data.get("tstop", 500.0))
            mean_rate = n_spikes / n_nodes / (tstop / 1000.0) if n_nodes > 0 else 0
            rows.append({
                "Simulator": sim_name,
                "Population": pop_name,
                "Neurons": n_nodes,
                "Total Spikes": n_spikes,
                "Active Neurons": n_active,
                "Mean Rate (Hz)": f"{mean_rate:.1f}",
            })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print()

    # ── 2. Voltage traces ─────────────────────────────────────────────
    n_sims = len(sim_results)
    fig, axes = plt.subplots(1, n_sims, figsize=(7 * n_sims, 5), squeeze=False)
    stim_start = 100.0
    stim_end = 500.0

    for col, (sim_name, pops) in enumerate(sorted(sim_results.items())):
        ax = axes[0, col]
        for pop_name, pdata in pops.items():
            vtraces = pdata.get("voltage_traces", {})
            for node_id, trace in sorted(vtraces.items(), key=lambda x: int(x[0])):
                times = np.array(trace["time"])
                vm = np.array(trace["voltage"])
                ax.plot(times, vm, label=f"Node {node_id}", alpha=0.8, linewidth=0.8)

            # Mark spike times
            spike_times = pdata.get("spike_times", [])
            spike_senders = pdata.get("spike_senders", [])
            for st, ss in zip(spike_times, spike_senders):
                ax.axvline(st, color="red", alpha=0.3, linewidth=0.5)

        ax.axvspan(stim_start, stim_end, alpha=0.1, color="blue", label="Stimulus")
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Membrane potential (mV)")
        ax.set_title(f"{sim_name} — Voltage Traces")
        ax.legend(fontsize=8, loc="upper right")
        ax.set_xlim(0, max(times) if len(times) else 500)

    plt.tight_layout()
    plt.savefig(str(output_root / "voltage_traces_comparison.png"), dpi=150)
    plt.show()
    print(f"Saved: {output_root / 'voltage_traces_comparison.png'}")
    print()

    # ── 3. Spike raster ───────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 4))
    colors = {"Brian2": "tab:blue", "NEST": "tab:orange"}
    y_offset = 0

    for sim_name, pops in sorted(sim_results.items()):
        for pop_name, pdata in pops.items():
            spike_times = pdata.get("spike_times", [])
            spike_senders = pdata.get("spike_senders", [])
            for st, ss in zip(spike_times, spike_senders):
                y = y_offset + int(ss)
                ax.scatter(st, y, color=colors.get(sim_name, "gray"), s=30, zorder=3)
            y_offset += max(spike_senders, default=0) + 2  # gap between sims

    ax.axvspan(stim_start, stim_end, alpha=0.1, color="blue")
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Neuron ID (offset per simulator)")
    ax.set_title("Spike Raster — NEST vs Brian2")
    ax.set_xlim(0, 500)

    # Legend
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:blue", markersize=8, label="Brian2"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="tab:orange", markersize=8, label="NEST"),
    ]
    ax.legend(handles=legend_elements, loc="upper right")

    plt.tight_layout()
    plt.savefig(str(output_root / "spike_raster_comparison.png"), dpi=150)
    plt.show()
    print(f"Saved: {output_root / 'spike_raster_comparison.png'}")
    print()

    # ── 4. Overlay comparison (if both NEST and Brian2 available) ──────
    if "NEST" in sim_results and "Brian2" in sim_results:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        pop_name = "S1nonbarrel_neurons"

        for node_idx, ax in enumerate(axes):
            for sim_name, color in [("Brian2", "tab:blue"), ("NEST", "tab:orange")]:
                pdata = sim_results[sim_name].get(pop_name, {})
                vtraces = pdata.get("voltage_traces", {})
                key = str(node_idx)
                if key in vtraces:
                    trace = vtraces[key]
                    times = np.array(trace["time"])
                    vm = np.array(trace["voltage"])
                    ax.plot(times, vm, color=color, label=sim_name, alpha=0.8, linewidth=1)

                    # Mark spikes for this node
                    spike_times = pdata.get("spike_times", [])
                    spike_senders = pdata.get("spike_senders", [])
                    for st, ss in zip(spike_times, spike_senders):
                        if int(ss) == node_idx:
                            ax.axvline(st, color=color, alpha=0.4, linewidth=0.5, linestyle="--")

            ax.axvspan(stim_start, stim_end, alpha=0.08, color="blue")
            ax.set_xlabel("Time (ms)")
            ax.set_ylabel("Vm (mV)")
            ax.set_title(f"Node {node_idx} — NEST vs Brian2 Overlay")
            ax.legend()
            ax.set_xlim(0, 500)

        plt.tight_layout()
        plt.savefig(str(output_root / "nest_brian2_overlay.png"), dpi=150)
        plt.show()
        print(f"Saved: {output_root / 'nest_brian2_overlay.png'}")

    # ── 5. Note about missing NEURON results ──────────────────────────
    print()
    print("=" * 70)
    print("NOTE: Detailed NEURON and single_compartment NEURON simulations")
    print("were not run in this notebook. To compare all four simulators:")
    print("  1. Run the detailed circuit with neurodamus/bluecellulab (NEURON)")
    print("  2. Run the simplified single-compartment circuit with neurodamus/bluecellulab (NEURON)")
    print("  3. Re-run this cell to include them in the plots")
    print("=" * 70)

---

## Notes

**Bundled Parameters:**  
The `sonata_simplify` package includes default simplification parameters for 11 standard BBP etypes. These bundled defaults are used automatically.

**Custom Parameters:**  
For circuits with custom etypes not covered by the defaults, you'll need to prepare custom `simplified_parameters.json` and call the pipeline directly.

**NEURON Mechanisms:**  
The `single_compartment` algorithm requires NEURON with the circuit's MOD files compiled. If you see mechanism errors, run:
```bash
cd /path/to/circuit
nrnivmodl mod
```

**Production Workflow:**  
In production, the TaskManager will:
1. Create a `CircuitSimplificationSingleConfig` from the campaign config
2. Stage the circuit from entitycore
3. Run the simplification in a container with pre-compiled mechanisms
4. Register the output circuit with derivation links to the parent